# ISRO Challenge 4: Route Resilience Training Pipeline

**Instructions:**
1. **CRITICAL:** You must add the actual dataset, NOT a notebook. On the right panel under 'Add Input', search for `balraj98/deepglobe-road-extraction-dataset` (it should have a 'Dataset' tag, not a 'Notebook' tag).
2. Under **Session Options**, set Accelerator to **GPU T4x2** or **P100**.
3. Run all cells. The trained weights will be saved to your `/kaggle/working/` directory.

In [1]:
import os
import cv2
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


### 1. Model Architecture (Deep CNN + Transformer + U-Net + Skips)

In [2]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        return self.relu(out)

class DeepCNNBackbone(nn.Module):
    def __init__(self, in_channels=3, base_ch=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_ch, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True)
        )
        self.stage1 = self._make_layer(base_ch, base_ch, blocks=3, stride=1)
        self.stage2 = self._make_layer(base_ch, base_ch * 2, blocks=4, stride=2)
        self.stage3 = self._make_layer(base_ch * 2, base_ch * 4, blocks=6, stride=2)
        self.stage4 = self._make_layer(base_ch * 4, base_ch * 8, blocks=3, stride=2)
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return s0, s1, s2, s3, s4

class TransformerBottleneck(nn.Module):
    def __init__(self, dim, feat_size, num_layers=8, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.dim = dim
        self.h, self.w = feat_size
        self.pos_embed = nn.Parameter(torch.zeros(1, dim, self.h, self.w))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim * mlp_ratio,
            dropout=0.1, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        b, c, h, w = x.shape
        if h != self.pos_embed.shape[2] or w != self.pos_embed.shape[3]:
            pos_emb = F.interpolate(self.pos_embed, size=(h, w), mode='bilinear', align_corners=False)
        else:
            pos_emb = self.pos_embed
        x = x + pos_emb
        tokens = x.flatten(2).transpose(1, 2)
        tokens = self.encoder(tokens)
        tokens = self.norm(tokens)
        return tokens.transpose(1, 2).reshape(b, c, h, w)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, global_skip_channels=0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv1 = ResidualBlock(in_channels + skip_channels + global_skip_channels, out_channels)
        self.conv2 = ResidualBlock(out_channels, out_channels)
    def forward(self, x, skip, global_skip=None):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=True)
        concat_list = [x, skip]
        if global_skip is not None:
            if global_skip.shape[-2:] != skip.shape[-2:]:
                global_skip_resized = F.interpolate(global_skip, size=skip.shape[-2:], mode="bilinear", align_corners=True)
            else:
                global_skip_resized = global_skip
            concat_list.append(global_skip_resized)
        x = torch.cat(concat_list, dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class FinalUpBlock(nn.Module):
    def __init__(self, in_channels, global_skip_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv1 = ResidualBlock(in_channels + global_skip_channels, out_channels)
        self.conv2 = ResidualBlock(out_channels, out_channels)
    def forward(self, x, global_skip):
        x = self.up(x)
        if global_skip is not None:
            global_skip_resized = F.interpolate(global_skip, size=x.shape[-2:], mode="bilinear", align_corners=True)
            x = torch.cat([x, global_skip_resized], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class DeepRoadSegModel(nn.Module):
    def __init__(self, in_channels=3, base_ch=64, input_size=(256, 256), num_transformer_layers=8, num_heads=8):
        super().__init__()
        self.backbone = DeepCNNBackbone(in_channels, base_ch)
        bottleneck_dim = base_ch * 8
        bottleneck_size = (input_size[0] // 16, input_size[1] // 16)
        self.transformer = TransformerBottleneck(
            dim=bottleneck_dim, feat_size=bottleneck_size, 
            num_layers=num_transformer_layers, num_heads=num_heads
        )
        self.up4 = DecoderBlock(base_ch * 8, base_ch * 4, base_ch * 4)
        self.up3 = DecoderBlock(base_ch * 4, base_ch * 2, base_ch * 2, global_skip_channels=bottleneck_dim)
        self.up2 = DecoderBlock(base_ch * 2, base_ch, base_ch, global_skip_channels=bottleneck_dim)
        self.up1 = FinalUpBlock(base_ch, bottleneck_dim, base_ch // 2)
        self.head = nn.Conv2d(base_ch // 2, 1, kernel_size=1)

    def forward(self, x, return_logits=False):
        s0, s1, s2, s3, s4 = self.backbone(x)
        bottleneck = self.transformer(s4)
        d4 = self.up4(bottleneck, s3)
        d3 = self.up3(d4, s2, global_skip=bottleneck)
        d2 = self.up2(d3, s1, global_skip=bottleneck)
        d1 = self.up1(d2, global_skip=bottleneck)
        logits = self.head(d1)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=True)
        if return_logits:
            return logits
        return torch.sigmoid(logits)

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        smooth = 1e-6
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_loss = 1 - (2. * intersection + smooth) / (union + smooth)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss.mean()

### 2. Dataset and Augmentation Pipeline

In [3]:
class DeepGlobeDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        # Use recursive globbing to find all images
        all_files = glob.glob(os.path.join(data_dir, '**', '*.*'), recursive=True)
        potential_images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png')) and 'mask' not in os.path.basename(f).lower()]
        
        self.images = []
        # Only add an image to the dataset if we can strictly find its corresponding mask on disk!
        for img_path in potential_images:
            img_name = os.path.basename(img_path)
            if 'sat.jpg' in img_name:
                mask_path = img_path.replace('sat.jpg', 'mask.png')
            elif '.jpg' in img_name:
                mask_path = img_path.replace('.jpg', '_mask.png')
            else:
                mask_path = img_path.replace('.png', '_mask.png')
                
            if os.path.exists(mask_path):
                self.images.append(img_path)
                
        if len(self.images) == 0:
            raise ValueError(f"No valid image-mask pairs found recursively in {data_dir}. Check directory structure!")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img_name = os.path.basename(img_path)
        
        if 'sat.jpg' in img_name:
            mask_path = img_path.replace('sat.jpg', 'mask.png')
        elif '.jpg' in img_name:
            mask_path = img_path.replace('.jpg', '_mask.png')
        else:
            mask_path = img_path.replace('.png', '_mask.png')
            
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            mask = mask.unsqueeze(0)
        else:
            mask = np.expand_dims(mask, axis=-1)
            
        return image, mask

def get_train_transforms(img_size=256):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.7),
        A.CoarseDropout(num_holes_range=(2, 10), hole_height_range=(10, 40), hole_width_range=(10, 40), p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

### 3. Training Loop (Saves Best Model Only)

In [4]:
# Intelligent Auto-Detection of the Training Folder
TRAIN_DIR = None

# Search through /kaggle/input for any folder containing 'train' with images inside it
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs:
        train_path = os.path.join(root, 'train')
        # Check if there are .jpg or .png files in the train directory
        if any(f.endswith('.jpg') or f.endswith('.png') for f in os.listdir(train_path)):
            TRAIN_DIR = train_path
            break

if TRAIN_DIR is None:
    # Fallback to root directories if 'train' folder doesn't exist explicitly
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.jpg') or f.endswith('.png') for f in files):
            TRAIN_DIR = root
            break

if TRAIN_DIR is None:
    raise ValueError("Could not find any directory containing images in /kaggle/input. Did you add the dataset correctly?")

print(f"Auto-detected training directory: {TRAIN_DIR}")

train_dataset = DeepGlobeDataset(TRAIN_DIR, transform=get_train_transforms(256))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

model = DeepRoadSegModel(in_channels=3, base_ch=64, input_size=(256, 256)).to(device)
criterion = BCEDiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 15
best_loss = float('inf')
print(f"Found {len(train_dataset)} actual image-mask pairs! Starting training on {device}...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images, return_logits=True)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")
    
    # Save ONLY the best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "/kaggle/working/best_deep_road_seg_model.pth")
        print(f"⭐ New best model found! Loss decreased to {best_loss:.4f}. Saved to /kaggle/working/best_deep_road_seg_model.pth")
    else:
        print(f"Loss did not improve from {best_loss:.4f}")


Auto-detected training directory: /kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train
Found 6226 actual image-mask pairs! Starting training on cuda...


Epoch 1/15:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 1 Average Loss: 0.5673
⭐ New best model found! Loss decreased to 0.5673. Saved to /kaggle/working/best_deep_road_seg_model.pth


Epoch 2/15:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 2 Average Loss: 0.5070
⭐ New best model found! Loss decreased to 0.5070. Saved to /kaggle/working/best_deep_road_seg_model.pth


...

Epoch 13 Average Loss: 0.3039
⭐ New best model found! Loss decreased to 0.3039. Saved to /kaggle/working/best_deep_road_seg_model.pth


Epoch 14/15:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 14 Average Loss: 0.3025
⭐ New best model found! Loss decreased to 0.3025. Saved to /kaggle/working/best_deep_road_seg_model.pth


Epoch 15/15:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 15 Average Loss: 0.2972
⭐ New best model found! Loss decreased to 0.2972. Saved to /kaggle/working/best_deep_road_seg_model.pth
